# Notebook 2 - Demonstracja REST API (FastAPI)

W tym notebooku pokazujemy korzystanie z chatbota przez REST API.

W normalnym uzyciu uruchamiamy serwer komenda:
```bash
uvicorn app.main:app --reload
```
i wolamy endpointy np. `curl`-em albo z dowolnego jezyka.

Tutaj uzywamy `fastapi.testclient.TestClient`, ktory pozwala wywolywac aplikacje
FastAPI w pamieci - to ten sam kod ktory dziala w produkcji, ale bez koniecznosci
osobnego procesu serwera.

Aby caly notebook dzialal bez Ollamy, podstawiamy fake LLM do menedzera sesji.

In [1]:
import sys, os, json, time, uuid
sys.path.insert(0, os.path.abspath(".."))

from fastapi.testclient import TestClient

from app.main import app, get_session_manager
from app.session_manager import SessionManager, Session
from app.config import settings
from app.chatbot import Chatbot
from app.llm_client import LLMClient


def make_session_manager():
    """Buduje SessionManager - z prawdziwym LLM jezeli sie da, inaczej fake."""
    try:
        LLMClient(settings).generate([{"role": "user", "content": "ping"}])
        print("Uzywam prawdziwego modelu:", settings.llm_model)
        return SessionManager(cfg=settings)
    except Exception as exc:
        print(f"Ollama niedostepna ({type(exc).__name__}). Uzywam fake LLM.")

        class FakeLLM:
            def generate(self, messages):
                last = next((m for m in reversed(messages) if m["role"] == "user"), {"content": ""})
                return f"[fake] Powiedziales: {last['content']}"

        class FakeManager(SessionManager):
            def create(self, system_prompt=None):
                sid = str(uuid.uuid4())
                bot = Chatbot(cfg=self.cfg, llm_client=FakeLLM(), system_prompt=system_prompt)
                with self._lock:
                    self._sessions[sid] = Session(bot=bot, created_at=time.time(), last_used_at=time.time())
                return sid

        return FakeManager(cfg=settings)


sm = make_session_manager()
app.dependency_overrides[get_session_manager] = lambda: sm
client = TestClient(app)
print("Klient API gotowy.")

Ollama niedostepna (APIConnectionError). Uzywam fake LLM.
Klient API gotowy.


## 1. Health check

In [2]:
print(client.get("/health").json())

{'status': 'ok', 'model': 'llama3.2'}


## 2. Utworzenie dwoch niezaleznych sesji

Pokazujemy ze API pozwala na rownoczesna obsluge wielu rozmow - kazda ma wlasna
historie, identyfikowana przez `session_id`.

In [3]:
sid_a = client.post("/sessions").json()["session_id"]
sid_b = client.post("/sessions").json()["session_id"]
print("Sesja A:", sid_a)
print("Sesja B:", sid_b)

2026-05-23 08:40:48 | INFO     | chatbot | Chatbot zainicjalizowany | model=llama3.2 | max_ctx=3000 tok | max_hist=20 msg


2026-05-23 08:40:48 | INFO     | chatbot | Chatbot zainicjalizowany | model=llama3.2 | max_ctx=3000 tok | max_hist=20 msg


Sesja A: a4c7a74a-748a-4820-b3c8-7a1153719b12
Sesja B: 1c9e2a39-dbd8-4b5c-bcea-a5afdd0dbcb7


## 3. Rozmowy rownolegle - kazda sesja ma swoja historie

In [4]:
client.post(f"/sessions/{sid_a}/chat", json={"message": "Mam na imie Anna."})
client.post(f"/sessions/{sid_b}/chat", json={"message": "Mam na imie Bartek."})

r_a = client.post(f"/sessions/{sid_a}/chat", json={"message": "Jak mam na imie?"}).json()
r_b = client.post(f"/sessions/{sid_b}/chat", json={"message": "Jak mam na imie?"}).json()

print("Sesja A:", r_a["reply"])
print("Sesja B:", r_b["reply"])

2026-05-23 08:40:48 | INFO     | chatbot | USER: Mam na imie Anna.


2026-05-23 08:40:48 | INFO     | chatbot | ASSISTANT: [fake] Powiedziales: Mam na imie Anna.


2026-05-23 08:40:48 | INFO     | chatbot | USER: Mam na imie Bartek.


2026-05-23 08:40:48 | INFO     | chatbot | ASSISTANT: [fake] Powiedziales: Mam na imie Bartek.


2026-05-23 08:40:48 | INFO     | chatbot | USER: Jak mam na imie?


2026-05-23 08:40:48 | INFO     | chatbot | ASSISTANT: [fake] Powiedziales: Jak mam na imie?


2026-05-23 08:40:48 | INFO     | chatbot | USER: Jak mam na imie?


2026-05-23 08:40:48 | INFO     | chatbot | ASSISTANT: [fake] Powiedziales: Jak mam na imie?


Sesja A: [fake] Powiedziales: Jak mam na imie?
Sesja B: [fake] Powiedziales: Jak mam na imie?


## 4. Pobranie historii rozmowy

In [5]:
history = client.get(f"/sessions/{sid_a}/history").json()
for m in history["messages"]:
    print(f"[{m['role']:>9}]: {m['content']}")

[     user]: Mam na imie Anna.
[assistant]: [fake] Powiedziales: Mam na imie Anna.
[     user]: Jak mam na imie?
[assistant]: [fake] Powiedziales: Jak mam na imie?


## 5. Walidacja wejscia - puste wiadomosci sa odrzucane przez Pydantica

In [6]:
resp = client.post(f"/sessions/{sid_a}/chat", json={"message": ""})
print("Status:", resp.status_code)
print("Body:", resp.json())

Status: 422
Body: {'detail': [{'type': 'string_too_short', 'loc': ['body', 'message'], 'msg': 'String should have at least 1 character', 'input': '', 'ctx': {'min_length': 1}}]}


## 6. 404 dla nieistniejacej sesji

In [7]:
resp = client.post("/sessions/nieistniejaca-sesja/chat", json={"message": "halo"})
print(resp.status_code, resp.json())

404 {'detail': 'Sesja nie istnieje'}


## 7. Lista sesji i sprzatanie

In [8]:
print("Aktywne sesje:", client.get("/sessions").json())
print("Usuwam sesje A:", client.delete(f"/sessions/{sid_a}").status_code)
print("Aktywne sesje:", client.get("/sessions").json())

2026-05-23 08:40:48 | INFO     | chatbot | Usunieto sesje a4c7a74a-748a-4820-b3c8-7a1153719b12


Aktywne sesje: {'sessions': ['a4c7a74a-748a-4820-b3c8-7a1153719b12', '1c9e2a39-dbd8-4b5c-bcea-a5afdd0dbcb7']}
Usuwam sesje A: 204
Aktywne sesje: {'sessions': ['1c9e2a39-dbd8-4b5c-bcea-a5afdd0dbcb7']}


## 8. Wywolanie API z zewnatrz (np. curl)

W normalnym uzyciu uruchamiamy serwer w terminalu:
```bash
uvicorn app.main:app --reload
```
i nastepnie:
```bash
SID=$(curl -s -X POST http://localhost:8000/sessions | jq -r .session_id)
curl -X POST http://localhost:8000/sessions/$SID/chat \
     -H "Content-Type: application/json" \
     -d '{"message": "Czesc, czy jestes botem?"}'
```

Dokumentacja interaktywna API jest pod `http://localhost:8000/docs`.